# NB_hospital_daily_summary — Gold Layer

Builds `adb_caresync.gold.hospital_daily_summary`: one row per hospital per calendar day with test-volume metrics, abnormal/critical rates, and day-over-day trend via LAG window. Full rebuild every run.

In [0]:
%sql
-- ============================================================
-- gold.hospital_daily_summary
-- Grain : one row per hospital per calendar day
-- Load  : FULL — CREATE OR REPLACE TABLE ... AS SELECT
-- Sources: silver.lab_results  (driving, grouped by hospital_id + test_date)
--          silver.hospitals     (LEFT JOIN on hospital_id)
--          silver.calendar      (LEFT JOIN on date_key for day attributes)
-- Window : LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)
-- ============================================================

CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS

WITH daily_agg AS (
  SELECT
    hospital_id,
    test_date                                                              AS summary_date,
    COUNT(*)                                                               AS total_tests,
    COUNT(DISTINCT patient_id)                                             AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                                              AS active_doctor_count,
    COUNT(*) FILTER (WHERE result_status = 'Abnormal')                     AS abnormal_test_count,
    COUNT(*) FILTER (WHERE result_status = 'Critical')                     AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),

enriched AS (
  SELECT
    d.hospital_id,
    h.hospital_name,
    d.summary_date,
    cal.day_name,
    cal.is_weekend,
    d.total_tests,
    d.unique_patients_tested,
    d.active_doctor_count,
    d.abnormal_test_count,
    d.critical_test_count,
    ROUND(CAST(d.abnormal_test_count AS DOUBLE) / d.total_tests * 100, 2)  AS abnormal_rate_pct,
    ROUND(CAST(d.critical_test_count AS DOUBLE) / d.total_tests * 100, 2)  AS critical_rate_pct,
    -- LAG window: previous day's total for this hospital
    LAG(d.total_tests) OVER (
      PARTITION BY d.hospital_id
      ORDER BY d.summary_date
    )                                                                      AS prev_day_total_tests
  FROM daily_agg d
  LEFT JOIN adb_caresync.silver.hospitals h
    ON d.hospital_id = h.hospital_id
  LEFT JOIN adb_caresync.silver.calendar cal
    ON CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT) = cal.date_key
)

SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  CAST(abnormal_rate_pct AS DECIMAL(5, 2))   AS abnormal_rate_pct,
  CAST(critical_rate_pct AS DECIMAL(5, 2))   AS critical_rate_pct,
  prev_day_total_tests,
  CAST(
    CASE
      WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
      ELSE ROUND(
             (total_tests - prev_day_total_tests)
             / CAST(prev_day_total_tests AS DOUBLE) * 100.0,
           2)
    END
  AS DECIMAL(6, 2))                          AS test_volume_change_pct

FROM enriched

In [0]:
record_count = spark.table("adb_caresync.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))